# M7 · Training well

**Outcome:** Compare training and validation behavior reproducibly.

Run cells with **Shift + Enter**. PyTorch is already installed in standard Colab runtimes.

## Experiment question

> What must be fixed before two runs are comparable?

Before running code, write a prediction. Then observe the evidence, change one variable, and explain the difference.

In [ ]:
import torch
print('PyTorch', torch.__version__)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import random, numpy as np
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
print('seeded:', seed)

## Diagnose with two curves
Training loss asks whether the model can fit; validation loss asks whether that fit transfers. Preserve the checkpoint at the best validation epoch, not necessarily the final epoch.

## Experiment record
Write down your hypothesis, code revision, data split, seed, model size, optimizer, learning rate, batch size, and both training and validation curves.

## Numerical and hardware limits
Mixed precision can accelerate eligible operations and reduce activation storage, but standard AMP commonly retains FP32 parameters and optimizer state. Measure examples per second and validation quality rather than assuming a speedup.

In [ ]:
P = 100_000_000                 # trainable parameters
micro_batch = 8
saved_elements_per_sample = 20_000_000
activation_bytes = 2            # FP16/BF16

weights = P * 4
gradients = P * 4
adam_moments = P * 8
activations = micro_batch * saved_elements_per_sample * activation_bytes
subtotal_gib = (weights + gradients + adam_moments + activations) / 1024**3
peak_plan_gib = subtotal_gib * 1.15
print('subtotal GiB:', round(subtotal_gib, 2))
print('with 15% reserve:', round(peak_plan_gib, 2))

In [ ]:
import math
micro_batch, devices, accumulation = 8, 4, 2
micro_step_ms, dataset_examples = 100, 100_000
global_batch = micro_batch * devices * accumulation
steps_per_epoch = math.ceil(dataset_examples / global_batch)
update_ms = accumulation * micro_step_ms
throughput = global_batch / (update_ms / 1000)
print('global batch:', global_batch)
print('steps/epoch:', steps_per_epoch)
print('ideal examples/s:', throughput)

## Quantization belongs in the right phase
Post-training quantization changes the exported inference model. Quantization-aware training simulates rounding during forward passes so parameters can adapt to the error.

## Reflection

1. What did you predict?
2. What evidence did the output provide?
3. Which one variable did you change?
4. How does the result connect to the lesson's mental model?